# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vuchau0802/run-starter-notebooks/blob/main/work/notebooks/w03_data_contract.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

I’ll use the March 2026 partition of fact_content_daily_performance. One source row represents one client’s page on one report date. One row in my feature frame represents one client’s page at the March 16 review date. I’ll build features from March 1–15 and check the outcome using March 16–30. March 31 is outside both windows. The output would help a content editor decide which pages to review first for a possible refresh.

In [5]:
%pip install -q duckdb

import duckdb
from google.colab import userdata

con = duckdb.connect()
token = userdata.get("HF_TOKEN")
con.execute(f"CREATE SECRET flyrank_hf (TYPE huggingface, TOKEN '{token}')")
del token

march = (
    "read_parquet('hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/month=2026-03/*.parquet')"
)

## 2. Fields: feature / label / context / excluded

Features: Early impressions, early clicks, early click-through rate, early days with impressions, and the share of early impressions recorded March 9–15.

Label: declined is 1 when impressions fall by more than 20% from March 1–15 to March 16–30, and 0 otherwise.

Context: Client and page IDs group daily rows into pages. The client ID also keeps each client’s pages together during evaluation. Report dates assign rows to windows, and gsc_data_available IS TRUE identifies rows with Search Console coverage. I require 15 usable days in each window and at least 100 impressions in the first window.

Excluded: March 16–30 metrics can be used to calculate the label. I exclude them from the model inputs because they arrive after the review date.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Verify it with queries (grain, counts, missing values, windows)

I check whether the March partition has duplicate or missing keys, how many rows it contains and which dates it covers, and how many rows have Search Console coverage.

In [7]:
grain = con.sql(f"""
    WITH daily_keys AS (
        SELECT report_date, client_hash_id, content_hash_id, COUNT(*) AS n
        FROM {march}
        GROUP BY report_date, client_hash_id, content_hash_id
    )
    SELECT
        COUNT(*) FILTER (WHERE n > 1) AS duplicate_keys,
        COALESCE(SUM(n) FILTER (
            WHERE report_date IS NULL
               OR client_hash_id IS NULL
               OR content_hash_id IS NULL
        ), 0) AS rows_with_missing_key
    FROM daily_keys
""").df()

display(grain)
assert grain.iloc[0].eq(0).all(), "Check the daily keys before building features."

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,duplicate_keys,rows_with_missing_key
0,0,0.0


In [8]:
display(con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        MIN(report_date) AS first_date,
        MAX(report_date) AS last_date
    FROM {march}
""").df())

,march_rows,first_date,last_date
0,9841378,2026-03-01,2026-03-31


In [9]:
display(con.sql(f"""
    SELECT
        COUNT(*) AS march_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS rows_kept,
        COUNT(*) FILTER (WHERE gsc_data_available IS FALSE) AS rows_flagged_false,
        COUNT(*) FILTER (WHERE gsc_data_available IS NULL) AS rows_with_unknown_flag
    FROM {march}
""").df())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,march_rows,rows_kept,rows_flagged_false,rows_with_unknown_flag
0,9841378,3611061,6230317,0


early_impressions is available at review because it sums recorded impressions from March 1–15.
early_clicks is available at review because it sums recorded clicks from March 1–15.
early_ctr_pct is available at review because both its clicks and impressions come from March 1–15.
early_visible_days is available at review because it counts March 1–15 days with at least one impression.
late_week_impression_share is available at review because it divides March 9–15 impressions by March 1–15 impressions.

In [10]:
pages = con.sql(f"""
    WITH page_windows AS (
        SELECT
            client_hash_id,
            content_hash_id,
            SUM(gsc_impressions) FILTER (
                WHERE report_date < DATE '2026-03-16'
            ) AS early_impressions,
            SUM(gsc_clicks) FILTER (
                WHERE report_date < DATE '2026-03-16'
            ) AS early_clicks,
            COUNT(*) FILTER (
                WHERE report_date < DATE '2026-03-16'
                  AND gsc_impressions > 0
            ) AS early_visible_days,
            SUM(gsc_impressions) FILTER (
                WHERE report_date BETWEEN DATE '2026-03-09' AND DATE '2026-03-15'
            ) AS late_week_impressions,
            SUM(gsc_impressions) FILTER (
                WHERE report_date >= DATE '2026-03-16'
            ) AS later_impressions
        FROM {march}
        WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-30'
          AND gsc_data_available IS TRUE
          AND gsc_impressions IS NOT NULL
          AND gsc_clicks IS NOT NULL
        GROUP BY client_hash_id, content_hash_id
        HAVING COUNT(DISTINCT report_date) FILTER (
            WHERE report_date < DATE '2026-03-16'
        ) = 15
        AND COUNT(DISTINCT report_date) FILTER (
            WHERE report_date >= DATE '2026-03-16'
        ) = 15
        AND early_impressions >= 100
    )
    SELECT
        client_hash_id,
        early_impressions,
        early_clicks,
        100.0 * early_clicks / early_impressions AS early_ctr_pct,
        early_visible_days,
        1.0 * late_week_impressions / early_impressions AS late_week_impression_share,
        CAST(5 * later_impressions < 4 * early_impressions AS INTEGER) AS declined
    FROM page_windows
    ORDER BY client_hash_id, content_hash_id
""").df()

feature_columns = [
    "early_impressions",
    "early_clicks",
    "early_ctr_pct",
    "early_visible_days",
    "late_week_impression_share",
]

X = pages[feature_columns].copy()
y = pages["declined"]
groups = pages["client_hash_id"]

print(f"Feature frame: {len(X):,} pages × {X.shape[1]} features")
display(X.describe().round(2))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Feature frame: 58,277 pages × 5 features


,early_impressions,early_clicks,early_ctr_pct,early_visible_days,late_week_impression_share
count,58277.00,58277.00,58277.00,58277.0,58277.00
mean,1896.19,5.50,0.28,15.0,0.45
std,3873.99,17.25,0.42,0.0,0.11
min,100.00,0.00,0.00,15.0,0.00
25%,299.00,0.00,0.00,15.0,0.39
50%,736.00,1.00,0.15,15.0,0.45
75%,1870.00,5.00,0.41,15.0,0.51
max,143173.00,1165.00,18.27,15.0,0.98


In [11]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import GroupShuffleSplit
from sklearn.tree import DecisionTreeClassifier

splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=groups))

assert set(groups.iloc[train_idx]).isdisjoint(groups.iloc[test_idx])
assert y.iloc[train_idx].nunique() == y.iloc[test_idx].nunique() == 2, (
    "ROC AUC requires both outcome classes in the train and test sets."
)

def held_out_auc(frame):
    model = DecisionTreeClassifier(max_depth=2, random_state=42)
    model.fit(frame.iloc[train_idx], y.iloc[train_idx])
    scores = model.predict_proba(frame.iloc[test_idx])[:, 1]
    return roc_auc_score(y.iloc[test_idx], scores)

# Copy the answer into a temporary feature frame to demonstrate leakage.
X_leaky = X.assign(copied_label=y)
leaky_auc = held_out_auc(X_leaky)
del X_leaky

honest_auc = held_out_auc(X)
assert X.columns.tolist() == feature_columns

print(f"ROC AUC with deliberate leak: {leaky_auc:.3f}")
print(f"ROC AUC with the five honest features: {honest_auc:.3f}")

ROC AUC with deliberate leak: 1.000
ROC AUC with the five honest features: 0.535


Copying the target into a temporary feature raised ROC AUC from 0.535 to 1.000 because the model could read the answer. I deleted the temporary frame and kept the five honest features. On this split, the honest score was only slightly above the chance level of 0.500.

## 4. Data limits

Requiring complete Search Console coverage in both windows excludes pages with reporting gaps. The results apply to eligible pages in this March slice. Measuring whether a refresh improves traffic would require a separate study.

In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.